<a href="https://colab.research.google.com/github/Boommook/cs539-botbuster/blob/main/consolidated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from pandas.core.dtypes import missing
## Load and inspect the dataset
df = pd.read_csv("../data/raw/twitter_human_bots_dataset.csv")

print("Shape:", df.shape)
df.head()
df.info()

print("Class counts: " + df["account_type"].value_counts())
print()
print(df["account_type"].value_counts(normalize=True))
"""
missing_cnts = df.isna().sum()
missing_prcnts = (df.isna().mean() * 100)
missing_summary = pd.DataFrame({
    "missing_count": missing_cnts,
    "missing_percentage": missing_prcnts
})

# sort the table to the cols w the most missing vals appear first
missing_summary.sort_values(by="missing_cnts", ascending=False)

# display only columns that contain at least one missing value.
display(missing_summary[missing_summary["missing_count"] > 0])
"""

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/twitter_human_bots_dataset.csv'

In [ ]:
df["account_type"].value_counts().plot(kind="bar")

plt.title("Account Type Distribution")
plt.xlabel("Account Type")
plt.ylabel("Number of Accounts")
plt.xticks(rotation=0)
plt.show()

Numerical Feature Distributions

In [ ]:
# I did this manually, but can be done dynamically too
numeric_cols = [
    "followers_count",
    "friends_count",
    "favourites_count",
    "statuses_count",
    "average_tweets_per_day"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flatten(), numeric_cols):
    # log1p reduces the effect of extremely large values
    ax.hist(np.log1p(df[col]), bins=50)
    ax.set_title(col)
    ax.set_xlabel("log(1 + value)")
    ax.set_ylabel("Count")

# need to hide unused graph I think

plt.tight_layout()
plt.show()

Numerical Feature Comparison btwn Humans and Bots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flatten(), numeric_cols):
    for label, group in df.groupby("account_type"):
        ax.hist(
            np.log1p(group[col]),
            bins=50,
            alpha=0.5,
            label=label
        )

    ax.set_title(col)
    ax.legend()

# I think this hides the unused graph (TO DO: ADD ABOVE TOO)
for ax in axes.flatten()[len(numeric_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

Compare Accounts Ages

In [ ]:
for label, group in df.groupby('account_type'):
    plt.hist(group['account_age_days'], bins=50, alpha=0.5, label=label)
plt.title('account_age_days by class')
plt.legend()
plt.show()

Box Plots by Type

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(
        data=df,
        x="account_type",
        y=np.log1p(df[col]),
        ax=ax
    )

    ax.set_title(col)
    ax.set_ylabel("log(1 + value)")

# hide unused graph
for ax in axes.flatten()[len(numeric_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

Examine Boolean Feature Relationships

In [ ]:
boolean_cols = [
    "default_profile",
    "default_profile_image",
    "geo_enabled",
    "verified"
]

for col in boolean_cols:
    print(f"\n{col}")

    display(
        pd.crosstab(
            df[col],
            df["account_type"],
            normalize="index"
        )
    )

Correlation Heatmap

In [ ]:
correlation_cols = numeric_cols + ["account_age_days"]

# we can change this if wanted
plt.figure(figsize=(10, 7))

sns.heatmap(
    df[correlation_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Numerical Feature Correlations")
plt.show()

Clean, Remove, and Add Features to the Dataset. Note: This could be broken lol

In [ ]:
def engineer_features(df):
  """
  Clean, remove, and add features to the dataset.

  Returens a copy of the dataset containing the newly engineered feats
  """
  model_df = df.copy()

  # convert the text target into int labels
  model_df["account_is_bot"] = (
        model_df["account_type"]
        .map({
            "human": 0,
            "bot": 1
        })
        .astype(int)
    )

  # ---------------------------------------------------------
  # Clean/remove features
  # ---------------------------------------------------------

  # remove columns that are identifiers, dupes, or unlikely to generalize
  model_df = model_df.drop(
        columns=[
            "Unnamed: 0",
            "id",
            "created_at",
            "profile_background_image_url",
            "profile_image_url"
        ],
        errors="ignore"
    )

  # ---------------------------------------------------------
  # Mssing-val features
  # ---------------------------------------------------------
  model_df["description_missing"] = model_df["description"].isna()

  model_df["location_unknown"] = (
    model_df["location"].isna()
    | model_df["location"].fillna("").str.lower().eq("unknown")
  )
  model_df["language_missing"] = df["lang"].isna()


  # text-based feats
  description = df["description"].fillna("")
  screen_name = df["screen_name"].fillna("")

  # Description features
  model_df["description_length"] = description.str.len()
  model_df["description_word_count"] = description.str.split().str.len()
  model_df["description_mention_count"] = description.str.count(r"@\w+")
  model_df["description_hashtag_count"] = description.str.count(r"#\w+")
  model_df["description_digit_count"] = description.str.count(r"\d")
  model_df["description_exclamation_count"] = description.str.count("!")

  # Screen-name features
  model_df["screen_name_length"] = screen_name.str.len()
  model_df["screen_name_digit_count"] = screen_name.str.count(r"\d")
  model_df["screen_name_underscore_count"] = screen_name.str.count("_")

  ## return completed dataframe
  return model_df


Final Check

In [ ]:
model_df = engineer_features(df)

print("Shape:", model_df.shape)

print("\nMissing values:")
print(model_df.isnull().sum().sort_values(ascending=False).head())

display(model_df.head())